# 02 - Feature Engineering e Governança dos Dados

Este notebook prepara a base de obesidade para modelagem, aplicando correções de qualidade, encoding das variáveis categóricas, mapeamento ordenado do alvo e divisão estratificada entre treino e teste.

In [10]:
from pathlib import Path

import joblib
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder

pd.set_option('display.max_columns', None)

RANDOM_STATE = 42
DATA_PATH = Path('../data/obesity.csv')
OUTPUT_DIR = Path('../data/processed')
ARTIFACT_DIR = Path('../models')

NOISY_COLUMNS = ['FCVC', 'NCP', 'CH2O', 'FAF', 'TUE']
TARGET_COL = 'Obesity_level'

## 1. Carga dos dados

A base original é carregada diretamente de `../data/obesity.csv`.

In [11]:
def load_dataset(path: Path) -> pd.DataFrame:
    """Carrega o dataset original de obesidade."""
    if not path.exists():
        raise FileNotFoundError(f'Arquivo não encontrado: {path}')

    return pd.read_csv(path)


df_raw = load_dataset(DATA_PATH)

print(f'Dimensões originais: {df_raw.shape}')
display(df_raw.head())

Dimensões originais: (2111, 17)


,Gender,Age,Height,Weight,family_history,FAVC,FCVC,NCP,CAEC,SMOKE,CH2O,SCC,FAF,TUE,CALC,MTRANS,Obesity
0,Female,21.0,1.62,64.0,yes,no,2.0,3.0,Sometimes,no,2.0,no,0.0,1.0,no,Public_Transportation,Normal_Weight
1,Female,21.0,1.52,56.0,yes,no,3.0,3.0,Sometimes,yes,3.0,yes,3.0,0.0,Sometimes,Public_Transportation,Normal_Weight
2,Male,23.0,1.80,77.0,yes,no,2.0,3.0,Sometimes,no,2.0,no,2.0,1.0,Frequently,Public_Transportation,Normal_Weight
3,Male,27.0,1.80,87.0,no,no,3.0,3.0,Sometimes,no,2.0,no,2.0,0.0,Frequently,Walking,Overweight_Level_I
4,Male,22.0,1.78,89.8,no,no,2.0,1.0,Sometimes,no,2.0,no,0.0,0.0,Sometimes,Public_Transportation,Overweight_Level_II


## 2. Validações iniciais de governança

Antes das transformações, validamos a presença da variável alvo, colunas esperadas, valores nulos e duplicados.

In [12]:
def validate_required_columns(df: pd.DataFrame, required_columns: list[str]) -> None:
    """Valida se todas as colunas obrigatórias existem no dataset."""
    missing_columns = [col for col in required_columns if col not in df.columns]
    if missing_columns:
        raise ValueError(f'Colunas obrigatórias ausentes: {missing_columns}')


print('Valores nulos por coluna:')
display(df_raw.isna().sum().sort_values(ascending=False).to_frame('missing_count'))

print(f'Linhas duplicadas: {df_raw.duplicated().sum()}')

Valores nulos por coluna:


,missing_count
Gender,0
Age,0
Height,0
Weight,0
family_history,0
FAVC,0
FCVC,0
NCP,0
CAEC,0
SMOKE,0


Linhas duplicadas: 24


## 3. Correção definitiva dos ruídos decimais

As variáveis `FCVC`, `NCP`, `CH2O`, `FAF` e `TUE` possuem ruídos decimais e devem retornar às suas escalas categóricas corretas por meio de `round()`.

In [13]:
def round_noisy_columns(df: pd.DataFrame, columns: list[str]) -> pd.DataFrame:
    """Aplica round() nas colunas com ruído decimal e converte para inteiro."""
    df = df.copy()

    for col in columns:
        df[col] = pd.to_numeric(df[col], errors='coerce').round().astype('Int64')

    return df


df_clean = round_noisy_columns(df_raw, NOISY_COLUMNS)

print('Amostra das colunas corrigidas:')
display(df_clean[NOISY_COLUMNS].head())

Amostra das colunas corrigidas:


,FCVC,NCP,CH2O,FAF,TUE
0,2,3,2,0,1
1,3,3,3,3,0
2,2,3,2,2,1
3,3,3,2,2,0
4,2,1,2,0,0


## 4. Mapeamentos de encoding

Aplicamos mapeamentos diretos para variáveis binárias, mapeamentos ordinais para variáveis com ordem natural e One-Hot Encoding para variável nominal sem ordem.

In [14]:
binary_mappings = {
    'Gender': {'Female': 0, 'Male': 1},
    'family_history': {'no': 0, 'yes': 1},
    'FAVC': {'no': 0, 'yes': 1},
    'SMOKE': {'no': 0, 'yes': 1},
    'SCC': {'no': 0, 'yes': 1},
}

ordinal_mappings = {
    'CAEC': {'no': 0, 'Sometimes': 1, 'Frequently': 2, 'Always': 3},
    'CALC': {'no': 0, 'Sometimes': 1, 'Frequently': 2, 'Always': 3},
}

target_mapping = {
    'Insufficient_Weight': 0,
    'Normal_Weight': 1,
    'Overweight_Level_I': 2,
    'Overweight_Level_II': 3,
    'Obesity_Type_I': 4,
    'Obesity_Type_II': 5,
    'Obesity_Type_III': 6,
}

nominal_features = ['MTRANS']

## 5. Aplicação dos encodings

A função abaixo centraliza as regras de transformação para garantir rastreabilidade e replicabilidade no pipeline de modelagem.

In [15]:
def apply_feature_engineering(
    df: pd.DataFrame,
    binary_mappings: dict,
    ordinal_mappings: dict,
    nominal_features: list[str],
    target_col: str,
    target_mapping: dict,
):
    """Aplica encoding nas variáveis explicativas e no alvo."""
    df = df.copy()

    for col, mapping in binary_mappings.items():
        if col in df.columns:
            df[col] = df[col].map(mapping).astype('Int64')

    for col, mapping in ordinal_mappings.items():
        if col in df.columns:
            df[col] = df[col].map(mapping).astype('Int64')

    if target_col not in df.columns:
        raise ValueError(f'Coluna alvo não encontrada: {target_col}')

    df[target_col] = df[target_col].map(target_mapping).astype('Int64')

    encoder = OneHotEncoder(handle_unknown='ignore', sparse_output=False)
    existing_nominal_features = [col for col in nominal_features if col in df.columns]

    if existing_nominal_features:
        encoded_array = encoder.fit_transform(df[existing_nominal_features])
        encoded_columns = encoder.get_feature_names_out(existing_nominal_features)
        encoded_df = pd.DataFrame(encoded_array, columns=encoded_columns, index=df.index).astype(int)
        df = pd.concat([df.drop(columns=existing_nominal_features), encoded_df], axis=1)
    else:
        encoder = None

    return df, encoder


df_encoded, nominal_encoder = apply_feature_engineering(
    df_clean,
    binary_mappings=binary_mappings,
    ordinal_mappings=ordinal_mappings,
    nominal_features=nominal_features,
    target_col="Obesity",
    target_mapping=target_mapping,
)

print(f'Dimensões após encoding: {df_encoded.shape}')
display(df_encoded.head())

Dimensões após encoding: (2111, 21)


,Gender,Age,Height,Weight,family_history,FAVC,FCVC,NCP,CAEC,SMOKE,CH2O,SCC,FAF,TUE,CALC,Obesity,MTRANS_Automobile,MTRANS_Bike,MTRANS_Motorbike,MTRANS_Public_Transportation,MTRANS_Walking
0,0,21.0,1.62,64.0,1,0,2,3,1,0,2,0,0,1,0,1,0,0,0,1,0
1,0,21.0,1.52,56.0,1,0,3,3,1,1,3,1,3,0,1,1,0,0,0,1,0
2,1,23.0,1.80,77.0,1,0,2,3,1,0,2,0,2,1,2,1,0,0,0,1,0
3,1,27.0,1.80,87.0,0,0,3,3,1,0,2,0,2,0,2,2,0,0,0,0,1
4,1,22.0,1.78,89.8,0,0,2,1,1,0,2,0,0,0,1,3,0,0,0,1,0


## 6. Validação pós-transformação

Validamos se o dataset final está numérico, sem nulos gerados pelos mapeamentos e pronto para treino.

In [16]:
null_counts = df_encoded.isna().sum()
columns_with_nulls = null_counts[null_counts > 0]

if not columns_with_nulls.empty:
    print('Atenção: colunas com nulos após transformação:')
    display(columns_with_nulls)
else:
    print('Nenhum valor nulo após transformação.')

non_numeric_columns = df_encoded.select_dtypes(exclude=['number']).columns.tolist()
print('Colunas não numéricas restantes:', non_numeric_columns)

display(df_encoded.dtypes.to_frame('dtype'))

Nenhum valor nulo após transformação.
Colunas não numéricas restantes: []


,dtype
Gender,Int64
Age,float64
Height,float64
Weight,float64
family_history,Int64
FAVC,Int64
FCVC,Int64
NCP,Int64
CAEC,Int64
SMOKE,Int64


## 7. Separação entre X e y

Separamos as variáveis independentes (`X`) da variável alvo (`y`), já mapeada para valores numéricos ordenados logicamente.

In [17]:
X = df_encoded.drop(columns=["Obesity"])
y = df_encoded["Obesity"].astype(int)

print('Shape de X:', X.shape)
print('Shape de y:', y.shape)

print('Distribuição do alvo:')
display(y.value_counts(normalize=True).sort_index().to_frame('proportion'))

Shape de X: (2111, 20)
Shape de y: (2111,)
Distribuição do alvo:


,proportion
Obesity,
0,0.128849
1,0.135955
2,0.137376
3,0.137376
4,0.166272
5,0.140692
6,0.153482


## 8. Divisão estratificada entre treino e teste

A divisão usa `stratify=y` para preservar a proporção das classes de obesidade nos conjuntos de treino e teste.

In [18]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=RANDOM_STATE,
    stratify=y
)

print('X_train:', X_train.shape)
print('X_test:', X_test.shape)
print('y_train:', y_train.shape)
print('y_test:', y_test.shape)

stratification_check = pd.DataFrame({
    'train_distribution': y_train.value_counts(normalize=True).sort_index(),
    'test_distribution': y_test.value_counts(normalize=True).sort_index(),
})

display(stratification_check)

X_train: (1688, 20)
X_test: (423, 20)
y_train: (1688,)
y_test: (423,)


,train_distribution,test_distribution
Obesity,,
0,0.129147,0.127660
1,0.135664,0.137116
2,0.137441,0.137116
3,0.137441,0.137116
4,0.166469,0.165485
5,0.140403,0.141844
6,0.153436,0.153664


## 9. Salvamento dos conjuntos e artefatos

Os conjuntos processados e os artefatos de mapeamento são salvos para permitir reprodutibilidade no treinamento e na aplicação.

In [19]:
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)

X_train.to_csv(OUTPUT_DIR / 'X_train.csv', index=False)
X_test.to_csv(OUTPUT_DIR / 'X_test.csv', index=False)
y_train.to_frame(TARGET_COL).to_csv(OUTPUT_DIR / 'y_train.csv', index=False)
y_test.to_frame(TARGET_COL).to_csv(OUTPUT_DIR / 'y_test.csv', index=False)
df_encoded.to_csv(OUTPUT_DIR / 'obesity_encoded.csv', index=False)

feature_engineering_artifact = {
    'binary_mappings': binary_mappings,
    'ordinal_mappings': ordinal_mappings,
    'target_mapping': target_mapping,
    'target_inverse_mapping': {v: k for k, v in target_mapping.items()},
    'nominal_features': nominal_features,
    'nominal_encoder': nominal_encoder,
    'feature_columns': X.columns.tolist(),
    'target_column': TARGET_COL,
    'noisy_columns': NOISY_COLUMNS,
}

joblib.dump(feature_engineering_artifact, ARTIFACT_DIR / 'feature_engineering_artifacts.pkl')

print('Arquivos salvos em ../data/processed/:')
print('- X_train.csv')
print('- X_test.csv')
print('- y_train.csv')
print('- y_test.csv')
print('- obesity_encoded.csv')
print('Artefatos salvos em ../models/feature_engineering_artifacts.pkl')

Arquivos salvos em ../data/processed/:
- X_train.csv
- X_test.csv
- y_train.csv
- y_test.csv
- obesity_encoded.csv
Artefatos salvos em ../models/feature_engineering_artifacts.pkl


## 10. Estrutura pronta para modelagem

Ao final deste notebook, os dados já estão corrigidos, codificados, separados e salvos de forma rastreável para uso direto no notebook de treinamento.